In [1]:
import numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.append('../../')
from utils.util import  *
from utils.mujoco_parser import MuJoCoParserClass

xml_path = '../../asset/ur5e/scene_ur5e_rg2_d435i_obj_realworld.xml'
env = MuJoCoParserClass(name='UR5e with RG2 gripper',rel_xml_path=xml_path,VERBOSE=True)
obj_names = [body_name for body_name in env.body_names
             if body_name is not None and (body_name.startswith("obj_"))]
n_obj = len(obj_names)
# Place objects
xyzs = sample_xyzs(
    n_sample=n_obj,x_range=[0.45,1.35],y_range=[-0.38,0.38],z_range=[0.74,0.74],min_dist=0.2,xy_margin=0.05)
colors = np.array([plt.cm.gist_rainbow(x) for x in np.linspace(0,1,n_obj)])
colors[:,3] = 1.0 # transparent objects
for obj_idx,obj_name in enumerate(obj_names):
    if obj_name == 'obj_target_01':
        continue
    jntadr = env.model.body(obj_name).jntadr[0]
    env.model.joint(jntadr).qpos0[:3] = xyzs[obj_idx,:]

# Move tables and robot base
env.model.body('base_table').pos = np.array([0,0,0.0])
env.model.body('front_object_table').pos = np.array([0.38+0.6,0,0])
env.model.body('base').pos = np.array([0.18,0,0.79])
for body_name in ['base_table','front_object_table']:
    geomadr = env.model.body(body_name).geomadr[0]
    env.model.geom(geomadr).rgba[3] = 1.0
print ("Ready.")

dt:[0.0020] HZ:[500]
n_dof (=nv):[60]
n_geom:[62]
geom_names:['floor', None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 'front_object_table', 'right_object_table', 'left_object_table', 'base_table', 'obj_cylinder_01', 'obj_cylinder_02', 'obj_cylinder_03', 'obj_cylinder_04', 'obj_cylinder_05', 'obj_cylinder_06', 'obj_cylinder_07', 'obj_cylinder_08']
n_body:[31]
body_names:['world', 'base', 'shoulder_link', 'upper_arm_link', 'forearm_link', 'wrist_1_link', 'wrist_2_link', 'wrist_3_link', 'tcp_link', 'camera_mount', 'd435i', 'rg2_gripper_base_link', 'camera_center', 'rg2_gripper_finger1_finger_link', 'rg2_gripper_finger1_inner_knuckle_link', 'rg2_gripper_finger1_finger_tip_link', 'rg2_gripper_finger2_finger_link', 'rg2_gripper_finger2_inner_knuckle

In [2]:
env.init_viewer(viewer_title='UR5e with RG2',viewer_width=1200,viewer_height=800,
                viewer_hide_menus=True)
env.update_viewer(azimuth=160,distance=4.00,elevation=-30,lookat=[0.2,-0.13,0.34],
                  VIS_TRANSPARENT=False,VIS_CONTACTPOINT=False,
                  contactwidth=0.05,contactheight=0.05,contactrgba=np.array([1,0,0,1]),
                  VIS_JOINT=False,jointlength=0.5,jointwidth=0.1,
                  jointrgba=[0.2,0.6,0.8,0.6])
env.reset()

init_pose = np.array([np.deg2rad(-90), np.deg2rad(-132.46), np.deg2rad(122.85), np.deg2rad(99.65), np.deg2rad(45), np.deg2rad(-90.02)])
env.forward(q=init_pose,joint_idxs=env.idxs_forward)

FIRST_FLAG = True
while env.is_viewer_alive():
    
    # Modify the properties of objects
    for obj_name in obj_names:
        if obj_name == 'obj_cylinder_01': continue
        geomadr = env.model.body(obj_name).geomadr[0]
        # change size
        geom_size = env.model.geom_size[geomadr,:]
        geom_size[0] = geom_size[0] + 0.0 # increase the radius of cylnder
        geom_size[1] = geom_size[1] + 0.0 # increase the height of cylnder
        env.model.geom_size[geomadr,:] = geom_size

        # change position (Random Disturbance)
        jntadr  = env.model.body(obj_name).jntadr[0]
        qposadr = env.model.jnt_qposadr[jntadr]
        geom_pos = env.data.qpos[qposadr:qposadr+3]
        geom_pos[:2] = geom_pos[:2] + 0.0001*np.random.randn(2)
        env.data.qpos[qposadr:qposadr+3] = geom_pos
        env.data.qpos[qposadr+3:qposadr+7] = r2quat(rpy2r(np.radians([0,0,0])))
        
    # Step
    env.step(ctrl=np.append(init_pose,1.0),ctrl_idxs=env.idxs_step+[6])
    
    # Render
    if env.loop_every(HZ=20) or FIRST_FLAG:
        # Get camera observation
        p_cam,R_cam = env.get_pR_body(body_name='camera_center')
        p_ego  = p_cam
        p_trgt = p_cam + R_cam[:,2]
        rgb_img,depth_img,pcd,xyz_img,xyz_img_world = env.get_egocentric_rgb_depth_pcd(
            p_ego=p_ego,p_trgt=p_trgt,rsz_rate=60,fovy=45,BACKUP_AND_RESTORE_VIEW=True)
        # for p in pcd: env.plot_sphere(p=p,r=0.005,rgba=[0.95,0.05,0.05,1])
        env.render(render_every=1)
    
    # Clear flag
    FIRST_FLAG = False

env.close_viewer()
print ("Done.")

Pressed ESC
Quitting.
Done.


2025-06-27 11:01:26.144 python[81385:14613207] TSM AdjustCapsLockLEDForKeyTransitionHandling - _ISSetPhysicalKeyboardCapsLockLED Inhibit


### `pip install 'pyvista[jupyter]'`

In [5]:
import time
import pyvista as pv
import numpy as np

start = time.time()
def backproject(depth_img, intrinsic_matrix, return_finite_depth=True, return_selection=False):

    depth = depth_img.astype(np.float32, copy=True)
    K = intrinsic_matrix
    K_inv = np.linalg.inv(K)

    # compute the 3D points
    width = depth.shape[1]
    height = depth.shape[0]

    # construct the 2D points matrix
    x, y = np.meshgrid(np.arange(width), np.arange(height))
    ones = np.ones((height, width), dtype=np.float32)
    x2d = np.stack((x, y, ones), axis=2).reshape(width*height, 3)

    # backprojection
    R = np.dot(K_inv, x2d.transpose())

    # compute the 3D points
    X = np.multiply(np.tile(depth.reshape(1, width*height), (3, 1)), R)
    X = np.array(X).transpose()
    if return_finite_depth:
        selection = np.isfinite(X[:, 0])
        X = X[selection, :]

    if return_selection:
        return X, selection
        
    return X

def draw_scene(pcd, pcd_color=None, plasma_coloring=False):
    plotter = pv.Plotter()
    if pcd_color is None:
        if plasma_coloring:
            plotter.add_points(pcd, scalars=pcd[:, 2], cmap='plasma')
        else:
            plotter.add_points(pcd, color=(0.1, 0.1, 1))
    else:

        if plasma_coloring:
            plotter.add_points(pcd, scalars=pcd_color[:, 0], cmap='plasma')
        else:
            # Normalize RGB values and add an Alpha channel
            pcd_color = np.c_[pcd_color / 255.0, np.ones(pcd_color.shape[0])]
            plotter.add_points(pcd, scalars=pcd_color, rgb=True)
    plotter.show()

# Camera intrinsic
img_height = depth_img.shape[0]
img_width = depth_img.shape[1]
fovy = 45
focal_scaling = 0.5*img_height/np.tan(fovy*np.pi/360)
K = np.array(((focal_scaling,0,img_width/2),
                    (0,focal_scaling,img_height/2),
                    (0,0,1)))

# Removing points that are farther than 1 meter or missing depth 
depth_img[depth_img == 0] = np.nan
depth_img[depth_img > 1] = np.nan
pcd, selection = backproject(depth_img, K, return_finite_depth=True, return_selection=True)
pcd_colors = rgb_img.copy()
pcd_colors = np.reshape(pcd_colors, [-1, 3])
pcd_colors = pcd_colors[selection, :]

draw_scene(
    pcd,
    pcd_color=pcd_colors,
    plasma_coloring=False
)

end = time.time()
print(f"Runtime of the program is {end - start}")

Widget(value='<iframe src="http://localhost:60596/index.html?ui=P_0x1568e4eb0_2&reconnect=auto" class="pyvista…

Runtime of the program is 2.8425710201263428


In [6]:
pv_plooter = pv.Plotter()
pv_plooter.add_points(pcd)
pv_plooter.show()

Widget(value='<iframe src="http://localhost:60596/index.html?ui=P_0x16fa5d690_3&reconnect=auto" class="pyvista…